# 02. Traditional ML: HOG and color-histogram baselines

**Role 2.** This notebook reports the traditional-ML experiment on the ASL Alphabet dataset. It uses the single manifest split and the shared loader from [01_data.ipynb](01_data.ipynb); it does not create another split or change label meanings.

The experiment compares logistic regression, random forest, and linear SVM on 10,005 versus 54,900 training images. Validation is used for model selection and the fixed test set is reported once. The result cells below load the saved shared-evaluator reports, comparison figures, and experiment log; they do not retrain the models. To reproduce training into a fresh output directory, configure `ASL_DATA_DIR` and run `python experiment1.py --results-dir results/experiment1_rerun` from the repository root. Trained wrappers are saved as `results/experiment1_<model>_<data-variant>_model.joblib`; run `python experiment1.py --export-models` to export wrappers matching an existing run. Each wrapper implements `predict(rgb_image) -> (letter, confidence)` for a single preprocessed 64 x 64 uint8 RGB image.

**Notebook contents**
1. Experimental methods and features
2. Shared evaluation and model comparison
3. Confusion analysis and limitations


## 1. Experimental results: traditional ML

### Goal and controlled comparison

This experiment compares logistic regression, a random forest, and a linear support vector machine (SVM) on the 29 ASL Alphabet classes. It also measures how training-set size affects each model. We use the shared, time-blocked manifest split described above rather than making a new random split: 54,900 full-training images, a 10,005-image subset, 9,225 validation images, and the same 8,875 test images for every model. The subset contains 345 training images per class; validation and test remain unchanged.

### Image preprocessing and features

The shared loader removes the 4-pixel border, resizes each RGB image to 64 x 64, and preserves the manifest labels. Features are extracted once and reused for both training-set sizes:

1. Convert each image to grayscale and compute Histogram of Oriented Gradients (HOG): 9 orientations, 16 x 16-pixel cells, 2 x 2-cell blocks, and L2-Hys normalization. This yields 324 features describing local hand edges and shape.
2. For each of the three RGB channels, compute an 8-bin histogram and divide bin counts by the number of pixels. This adds 24 color-distribution features.
3. Concatenate the two parts into one 348-feature vector. The resulting feature matrices are shared by all three models.

Logistic regression uses `max_iter=1000`; the random forest uses 100 trees; the SVM is `LinearSVC` with `max_iter=5000`. All use seed 42. Logistic regression and SVM standardize features using statistics fit on training data only; the forest uses unscaled features. These settings were fixed for the comparison, not tuned on the test set.

### Evaluation protocol

Each fitted model predicts the unchanged validation and test sets. The shared `evaluate.py` scorer reports accuracy, macro- and support-weighted precision, recall and F1, per-class scores, and a confusion matrix in the fixed class order. Training duration is recorded. CPU inference latency is measured on 200 evenly spaced, already-preprocessed test images, one image at a time; it includes HOG and color-histogram extraction plus prediction. The saved model wrapper accepts one 64 x 64 uint8 RGB image from the shared preprocessing and returns `(letter, confidence)`. Logistic regression and random forest use their predicted-class probabilities; the SVM confidence is a softmax of decision scores and is not calibrated. These timings depend on the machine and are not end-to-end webcam latency.

The cells below load the saved reports and artifacts from `results/`. Re-running this results section does not retrain the classifiers. To reproduce training, first make sure `ASL_DATA_DIR` points to the image dataset and run `python experiment1.py` from the repository root.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import evaluate

RESULTS_DIR = Path("results")
ARTIFACT_DIR = RESULTS_DIR / "experiment1"
comparison = pd.read_csv(ARTIFACT_DIR / "comparison.csv")
comparison = comparison.rename(columns={
    "val_accuracy": "validation_accuracy",
    "test_macro_precision": "macro_precision",
    "test_macro_recall": "macro_recall",
    "test_macro_f1": "macro_f1",
})

assert len(comparison) == 6
assert set(comparison["n_train"]) == {10005, 54900}
assert comparison["test_accuracy"].between(0, 1).all()

display(comparison[[
    "model", "training_size", "n_train", "validation_accuracy", "test_accuracy",
    "macro_precision", "macro_recall", "macro_f1", "train_time_s", "inference_ms"
]].style.format({
    "validation_accuracy": "{:.2%}", "test_accuracy": "{:.2%}",
    "macro_precision": "{:.2%}", "macro_recall": "{:.2%}", "macro_f1": "{:.2%}",
    "train_time_s": "{:.1f}", "inference_ms": "{:.2f}"
}))

# Keep this run in the shared log used by the experiment table and slides.
experiment_log = pd.read_csv(RESULTS_DIR / "experiment_log.csv")
experiment1_log = experiment_log[experiment_log["run_id"].str.startswith("experiment1_")]
display(experiment1_log)

size_test = comparison.pivot(index="model", columns="training_size", values="test_accuracy")
size_test["full_minus_10k_percentage_points"] = (size_test["full"] - size_test["10k"]) * 100
print("Test-accuracy change from 10k to full training data:")
display(size_test)

best_validation = comparison.loc[comparison["validation_accuracy"].idxmax()]
best_test = comparison.loc[comparison["test_accuracy"].idxmax()]
print(f"Highest validation accuracy: {best_validation['model']} ({best_validation['training_size']}, {best_validation['validation_accuracy']:.2%})")
print(f"Highest test accuracy (report only; not used for model selection): {best_test['model']} ({best_test['training_size']}, {best_test['test_accuracy']:.2%})")


### Comparison and size-test results

Each row uses the same fixed test set. Accuracy is the fraction of correct predictions; macro precision, recall and F1 average per-letter scores equally, so a class with fewer test frames is not hidden by frequent classes. The shared `results/experiment_log.csv` is the run log and source for the experiment table; the code cell above loads it alongside the saved JSON reports. Metrics are not manually retyped into another spreadsheet.

The table below compares validation and test metrics, training time and batch-one CPU latency for all six runs. The shared chart is generated from these saved reports, and the next code cell can recreate it from the comparison table.

![Experiment 1 comparison chart](results/experiment1/comparison.png)

Use validation results for model selection, not the held-out test scores. Each row is a single fixed-parameter run, not an uncertainty estimate. Inference latency depends on the CPU and machine load.


In [ ]:
labels = comparison["model"] + "\n" + comparison["training_size"]
positions = np.arange(len(comparison))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for offset, (column, label) in enumerate((
    ("test_accuracy", "Accuracy"),
    ("macro_precision", "Macro precision"),
    ("macro_recall", "Macro recall"),
    ("macro_f1", "Macro F1"),
)):
    axes[0].bar(positions + (offset - 1.5) * 0.2, comparison[column], width=0.2, label=label)
axes[0].set_xticks(positions, labels)
axes[0].set_ylim(0, 1)
axes[0].set_title("Test metrics")
axes[0].legend()

for ax, column, title in zip(
    axes[1:], ("train_time_s", "inference_ms"),
    ("Training time (seconds)", "CPU batch-one inference (ms/image)"),
):
    ax.bar(labels, comparison[column])
    ax.set_title(title)
for ax in axes:
    ax.tick_params(axis="x", labelrotation=35)
fig.tight_layout()
plt.show()

### Error analysis: which letters are confused?

The full-data random forest has the highest test accuracy in this comparison. The table and normalized confusion matrix below are generated from its shared test report. The confusion pairs are directed (true letter to predicted letter); the matrix rows are normalized by true class, so the diagonal represents per-letter recall. Lighter boxes on the diagonal had more prediction loss, whereas darker boxes had more prediction accuracy.

![Full-data random forest normalized test confusion matrix](results/experiment1/random_forest_full_confusion.png)

These are video frames, not independent signers, and neighboring frames can show nearly the same pose. A high count is therefore not the same as independent evidence about generalization. The saved prediction CSV links every result back to its original image path.


In [ ]:
report_path = RESULTS_DIR / "experiment1_random_forest_full_test.json"
rf_test = json.loads(report_path.read_text(encoding="utf-8"))

print("Most frequent directed confusions (full-data random forest):")
display(evaluate.top_confusions(rf_test, k=10))

per_class = pd.DataFrame(rf_test["per_class"]).T
per_class.index.name = "letter"
print("Lowest-recall classes:")
display(per_class.sort_values("recall").head(8).style.format({"precision": "{:.1%}", "recall": "{:.1%}", "f1": "{:.1%}", "support": "{:.0f}"}))

conditions = pd.read_csv(ARTIFACT_DIR / "random_forest_full_conditions.csv")
lighting = conditions.loc[conditions["condition"] == "lighting"].copy()
lighting["error_rate"] = lighting["error_rate"].astype(float)
print("Error rates by lighting and image-quality flags:")
display(conditions[["condition", "value", "count", "errors", "error_rate"]].style.format({"error_rate": "{:.1%}"}))

ax = lighting.set_index("value")["error_rate"].plot.bar(
    color=["#f58518", "#4c78a8", "#54a24b"],
    title="Full-data random forest: test error by lighting condition",
    ylabel="Misclassification rate", xlabel="Lighting",
    ylim=(0, 1), rot=0,
)
ax.figure.tight_layout()
plt.show()

### Interpreting the errors and limitations

A visual check of representative misclassified frames suggests plausible reasons, not proven causes. The D example has a prominent curled-finger loop that can resemble O. Some M -> A examples come from dim blocks, where the fist is dark and individual fingers are partly occluded. G/H and U/V differ in finger orientation or spacing; small shape details may not survive 64 x 64 pixels, coarse HOG cells, and a global color histogram. Representative paths include `D/D2580.jpg`, `M/M2328.jpg`, `G/G1608.jpg`, `U/U1136.jpg`, and `V/V1292.jpg`; all predictions and paths are in `results/experiment1/random_forest_full_predictions.csv`.

The grouped error rates by lighting and perceptual-hash flag are shown in the previous cell. They are associations, not causal experiments: lighting and recording blocks are related, and perceptual-hash overlap may mark inherently similar poses. The blur-flag sample is too small for a reliable conclusion.

All images come from one signer and room, even though the time-block split prevents neighboring frames from leaking across train and test. Therefore, these scores do not estimate performance on new signers or webcam conditions. A useful next experiment is to test lighting augmentation and finer shape features using validation results, then evaluate once on a separate set collected from new signers and rooms. The full error-pair rankings, condition summaries, per-image predictions, and model reports are saved under `results/experiment1/` and `results/`.